# 第 13 课 · 反爬与合规红线

> **内核提醒（每课都要看）**：本 notebook 必须在 conda 环境 **ai_learn** 中运行。
> 右上角内核应显示 `Python 3.10.21 ('ai_learn')`；打开后从上到下逐格运行（Shift+Enter）。
> 若报 `ModuleNotFoundError`，先检查内核是否选成了 ai_learn。

> 这是整个模块**最重要的一课**。技术决定你“能不能抓到”，合规决定你“该不该抓、抓了会不会出事”。
> 本课只讲清楚：**别人怎么防你、你该守哪些规矩**。

In [1]:
# 每本 notebook 开头都要重新 import
from pathlib import Path
from urllib.robotparser import RobotFileParser     # Python 自带，用来读 robots.txt
import time

DATA_DIR = Path(r'C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\08_爬虫\data')
print('数据目录存在吗：', DATA_DIR.exists())


数据目录存在吗： True


## 13.1 先立一条铁律：技术能做 ≠ 应该做

**生活化类比**：会开车不代表可以闯红灯。交规不是为了妨碍你，而是让你能长期、安全地开下去。

> **“我写得出来绕过”不是“我应该绕过去”的理由。**

对方网站做反爬，本身就是在表达“我不希望被这样采集”。
你可以学习它的**原理**（理解攻防双方怎么想），但不要把它当成“攻击授权书”。

## 13.2 常见反爬手段，以及“技术边界”在哪

| 手段 | 网站在干什么 | 合规的做法 | 越界的做法（别做） |
|------|--------------|------------|--------------------|
| 查 UA | 看你是不是 python | 写一个像浏览器的请求头 | 批量伪造 UA 装成很多真人 |
| 限速 | 看你是不是机器 | 两次请求之间等几秒 | 被提示太快还硬刷 |
| 代理 IP | 单 IP 高频就封 | 个人学习基本不需要 | 买代理池规避封禁 |
| 验证码 | 区分人和机器 | **停手**，尊重它 | 用打码平台自动绕过 |
| 登录态 | 要登录才给看 | 自己注册、按条款用 | 撞库、盗号、批量注册 |

> **判断口诀**：一旦需要“伪装成一个不存在的真人”“绕过人机验证”“突破登录墙”，
> 你就已经从“采集公开信息”滑向“未授权访问”了，应当停手。

## 13.3 robots.txt：网站门口的告示牌

**是什么**：很多网站根目录放一个 `robots.txt` 文件，用简单规则告诉爬虫：哪些路径允许抓、哪些禁止。规范叫 **RFC 9309**。

**生活化类比**：它就像店门口贴的“本店谢绝自带酒水/谢绝采访”告示。虽然不是法律，但它**清楚表达了店主的意愿**。

**法律地位要清醒**：robots.txt 本身不是法律，但无视它，往往会在后续纠纷里成为“你明知故犯”的证据。
合规的工程习惯是：**默认遵守**。

## 13.4 动手：用 Python 读本地 robots.txt 夹具

我们用标准库 `urllib.robotparser` 读本地 `robots_sample.txt`，
用 `can_fetch(UA, 路径)` 判断“某个网址允不允许抓”。

In [2]:
rp = RobotFileParser()
robots_text = (DATA_DIR / 'robots_sample.txt').read_text(encoding='utf-8')
rp.parse(robots_text.splitlines())      # 把告示牌逐行读进去

# 一条条问：这个 UA 能不能抓这个路径？
tests = [
    ('*', '/catalogue/page-1.html'),   # 公开的书籍列表页
    ('*', '/admin/users'),              # 后台管理
    ('*', '/checkout/order'),           # 结算页
    ('BadBot', '/catalogue'),            # 被点名封杀的 UA
]
for ua, path in tests:
    ok = rp.can_fetch(ua, path)
    print(f'UA={ua:8s} 抓 {path:26s} -> {"允许" if ok else "禁止"}')


UA=*        抓 /catalogue/page-1.html     -> 允许
UA=*        抓 /admin/users               -> 禁止
UA=*        抓 /checkout/order            -> 禁止
UA=BadBot   抓 /catalogue                 -> 禁止


> 在 Scrapy 里把 `ROBOTSTXT_OBEY = True` 打开（上一课 settings 里那行），
> 框架就会自动帮你做这个判断。真实项目第一步永远是：**先读 robots.txt，再决定抓不抓。**

## 13.5 对服务器“温柔点”：限速 time.sleep

**为什么**：爬虫是寄生在别人服务器上的。你一秒发 50 个请求，可能让小服务器卡顿，这等于搞破坏。

**怎么做**：两次请求之间主动停一下。`time.sleep(秒数)` 就是“让程序等一会儿”。

In [3]:
print('开始抓...')
for i in range(3):
    # 这里假装去请求一个页面；真实情况是 requests.get(...)
    print(f'  第 {i+1} 次请求')
    time.sleep(1)        # 每次请求后停 1 秒，对服务器温柔一点
print('结束。刚才一共等了约 2 秒。')


开始抓...
  第 1 次请求


  第 2 次请求


  第 3 次请求


结束。刚才一共等了约 2 秒。


## 13.6 UA：让请求“看起来像正常浏览器”

`requests` 默认会自报家门：`User-Agent: python-requests/2.x`。很多站点一看就知道是脚本，直接拒绝。
合规的做法：在请求头里写一个像 Chrome 的 UA（这是“让请求正常”，不是“伪装真人骗验证”）。
> 你自己的 `douban.py` 抓豆瓣时如果拿到非 200 状态码，往往就是因为没带浏览器 UA——这是正常现象，加个请求头即可，**不是要去突破什么**。

## 13.7 服务条款：比告示牌更硬的合同

比 robots.txt 更硬的是网站的**用户协议**。很多站明确写了“禁止自动抓取、禁止商用再利用”。
- 它是你和网站之间的合同，违反可能构成违约。
- **拿不准时就别抓，或者发邮件问站长授权。**
- 个人学习，最稳妥的就是用官方专门提供的练习站（books.toscrape / quotes.toscrape）。

## 13.8 法律边界（学习层面提示，非法律意见）

- **《个人信息保护法》**：姓名、手机号、身份证、人脸、行踪等是个人信息，**红线中的红线**，不要抓。
- **《数据安全法》**：重要数据的采集、存储有要求。
- **《网络安全法》**：绕登录、绕访问控制去拿数据，可能构成“非法获取计算机信息系统数据”。
- **《著作权法》**：文章图片有版权，批量爬取后公开转载、商用风险极高。

> 一句话：**抓公开的、非个人的、无条款禁止的数据，仅用于个人学习，风险最低；
> 碰个人信息、绕登录、商用，风险陡增。**

## 13.9 动手前自检清单（每次开抓过一遍）

1. 这个站的 robots.txt 允许我抓的路径吗？
2. 它的服务条款禁止自动抓取吗？
3. 我要的数据里**有没有个人信息**？（有就停）
4. 需要绕登录/验证码/访问控制吗？（需要就停）
5. 加了 `time.sleep` 限速和并发上限吗？
6. 如果站长明天找到我，我能解释清楚、且不心虚吗？

第 6 条是终极标准。

## 13.10 小结

- 技术能做 ≠ 合规该做；反爬原理是用来理解攻防，不是绕过授权。
- robots.txt（RFC 9309）+ 服务条款 = 网站主的明确意愿，默认遵守。
- 个人信息是红线；绕登录/绕验证可能触犯网络安全法。
- 对服务器保持克制：`time.sleep` 限速、见好就收。

## 13.11 随堂小练习（自己动手，空格里写）

1. 打开本地 `robots_sample.txt`，自己加一条 `Disallow: /secret/`，再用上面的 `rp.can_fetch('*', '/secret/')` 验证结果变成“禁止”。
2. 用一句话写下：你打算以后只在哪些站点上练手？

In [4]:
# TODO：自己写代码完成上面两题（随堂小练，没有单独答案文件）


## 来源与延伸

- Robots 协议 RFC 9309（标准原文）：https://www.rfc-editor.org/rfc/rfc9309
- Python urllib.robotparser 文档：https://docs.python.org/3/library/urllib.robotparser.html
- 《个人信息保护法》《网络安全法》《数据安全法》（全国人大官网）：http://www.npc.gov.cn/
- 官方练习站（放心练）：https://books.toscrape.com/ 、https://quotes.toscrape.com/
